# StudyLens: full evaluation on Google Colab

Runs the AI2D evaluations (and optionally the synthetic pilot) on a Colab GPU. All runs are **resumable**: caches live on
your Google Drive, so after a disconnect just re-run the cells.

* **GPU:** Runtime → Change runtime type → GPU (T4 works in fp16; L4/A100 are faster and use bf16).
* **Licences:** AI2D is under the AI2 non-commercial licence (no redistribution); keep the Drive folder private.
  Qwen2.5-VL-3B is under the Qwen Research License (non-commercial).
* **Consistency:** compare conditions only within one environment (CUDA vs Apple MPS outputs can differ slightly).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/studylens'   # caches + results are kept here
!mkdir -p $DRIVE

## 1. Get the code
Either clone your repository (replace the placeholder) or upload a zip of the project to Drive.

In [ ]:
REPO_URL = 'https://github.com/krish-patel1003/CECS558-term-project-studylens.git'
import os
if not os.path.exists('/content/studylens'):
    if REPO_URL.startswith('http'):
        !git clone $REPO_URL /content/studylens
    else:
        !unzip -q $DRIVE/studylens.zip -d /content/   # alternative: upload the project zip to Drive
%cd /content/studylens

In [ ]:
!pip install -q -r requirements.txt -r requirements-model.txt
# PaddleOCR (E1 baseline + scene-graph extraction) in its own venv: its numpy/PyYAML pins conflict with torch
!python -m venv /content/venv-ocr && /content/venv-ocr/bin/pip install -q -r requirements-ocr.txt
import os
os.environ['OCR_PYTHON'] = '/content/venv-ocr/bin/python'
os.environ['STUDYLENS_DEVICE'] = 'cuda'

## 2. Data and caches on Drive
`external/` (AI2D, caches) is linked to Drive so results survive disconnects. The AI2D download requires accepting its licence.

In [ ]:
!mkdir -p $DRIVE/external
!rm -rf external && ln -s $DRIVE/external external
!python scripts/get_ai2d.py --accept-license   # downloads AI2D into external/ai2d (once)

## 3. (Optional) LoRA adapter
Copy `checkpoints/ai2d_lora_r16/` to `$DRIVE/checkpoints/` from your laptop, or train it here:
`!python scripts/train_lora.py` (much faster on a GPU).

In [ ]:
!mkdir -p $DRIVE/checkpoints && rm -rf checkpoints && ln -s $DRIVE/checkpoints checkpoints
!ls checkpoints

## 4. Run the evaluations (resumable)
* `evaluate_grounding.py`: grounded explanations (StudyLens vs the same VLM with native grounding) + multiple choice.
* `evaluate_ai2d.py`: E1 / E2 / E3 conditions on the AI2D test split. `--n-diagrams 981` is the whole test split.

In [ ]:
!python -u scripts/evaluate_grounding.py --n-parts 150 --n-flow 100 --q-per-diagram 4

In [ ]:
!python -u scripts/evaluate_ai2d.py --adapter checkpoints/ai2d_lora_r16 --n-diagrams 981 --q-per-diagram 8

## 5. Results
`results/*.json` contain ids and metrics only. Copy them back into your local repository's `results/` folder; the app's
Evaluation tab reads them.

In [ ]:
!mkdir -p $DRIVE/results && cp results/*.json $DRIVE/results/ && ls -la $DRIVE/results